# Train Expression Recognition Model on Kaggle

Notebook này chạy lại training từ `train_exp.py` trên Kaggle.

**Điều kiện:**
- Dataset folder 92 phải được upload như Kaggle dataset
- GPU phải được enable
- Tất cả paths tự động thích ứng với Kaggle environment

## 0. Setup Environment

In [ ]:
import os
import sys
import torch
import numpy as np
import math
import logging
from datetime import datetime

print("=" * 80)
print("TRAINING ENVIRONMENT SETUP")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Working directory: {os.getcwd()}")
print(f"Current files: {os.listdir('.')[:15]}")
print("=" * 80)

In [ ]:
# Setup Kaggle paths
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Kaggle environment - dataset is in /kaggle/input
    BASE_PATH = '/kaggle/input/92-dataset/92'
    OUTPUT_PATH = '/kaggle/working/results'
    print(f"🎯 Running on Kaggle")
else:
    # Local environment
    BASE_PATH = '.'
    OUTPUT_PATH = './results'
    print(f"🎯 Running locally")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Add BASE_PATH to sys.path so we can import modules
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)

print(f"\nPaths configured:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  OUTPUT_PATH: {OUTPUT_PATH}")

# Verify dataset exists
if os.path.exists(BASE_PATH):
    print(f"✓ Dataset path verified")
    dataset_contents = os.listdir(BASE_PATH)
    print(f"  Contents: {', '.join(dataset_contents[:8])}...")
else:
    print(f"✗ ERROR: Dataset path not found: {BASE_PATH}")
    raise FileNotFoundError(f"Dataset not found at {BASE_PATH}")

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch import distributed
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

## 1. Custom Loss Functions (from train_exp.py)

In [ ]:
# Define custom losses from train_exp.py

def ACLoss(att_map1, att_map2, grid_l, output):
    """Attention Consistency Loss - compares attention maps of original and flipped images"""
    flip_grid_large = grid_l.expand(output.size(0), -1, -1, -1)
    flip_grid_large = Variable(flip_grid_large, requires_grad=False)
    flip_grid_large = flip_grid_large.permute(0, 2, 3, 1)
    att_map2_flip = F.grid_sample(att_map2, flip_grid_large, mode='bilinear', 
                                   padding_mode='border', align_corners=True)
    flip_loss_l = F.mse_loss(att_map1, att_map2_flip, reduction='none')
    return flip_loss_l


def generate_flip_grid(w, h):
    """Generate flip grid for attention map comparison"""
    x_ = torch.arange(w).view(1, -1).expand(h, -1)
    y_ = torch.arange(h).view(-1, 1).expand(-1, w)
    grid = torch.stack([x_, y_], dim=0).float()
    grid = grid.unsqueeze(0).expand(1, -1, -1, -1)
    grid[:, 0, :, :] = 2 * grid[:, 0, :, :] / (w - 1) - 1
    grid[:, 1, :, :] = 2 * grid[:, 1, :, :] / (h - 1) - 1
    grid[:, 0, :, :] = -grid[:, 0, :, :]
    return grid


class LSR2(nn.Module):
    """Label Smoothing with Relative weights"""
    def __init__(self, e):
        super().__init__()
        self.log_softmax = nn.LogSoftmax(dim=1)
        self.e = e

    def _one_hot(self, labels, classes, value=1):
        one_hot = torch.zeros(labels.size(0), classes)
        labels = labels.view(labels.size(0), -1)
        value_added = torch.Tensor(labels.size(0), 1).fill_(value)
        value_added = value_added.to(labels.device)
        one_hot = one_hot.to(labels.device)
        one_hot.scatter_add_(1, labels, value_added)
        return one_hot

    def _smooth_label(self, target, length, smooth_factor):
        one_hot = self._one_hot(target, length, value=1 - smooth_factor)
        mask = (one_hot == 0)
        balance_weight = torch.tensor([0.95124031, 4.36690391, 1.71143654, 0.25714585, 
                                       0.6191221, 1.74056738, 0.48617274]).to(one_hot.device)
        ex_weight = balance_weight.expand(one_hot.size(0), -1)
        resize_weight = ex_weight[mask].view(one_hot.size(0), -1)
        resize_weight /= resize_weight.sum(dim=1, keepdim=True)
        one_hot[mask] += (resize_weight * smooth_factor).view(-1)
        return one_hot.to(target.device)

    def forward(self, x, target):
        smoothed_target = self._smooth_label(target, x.size(1), self.e)
        x = self.log_softmax(x)
        loss = torch.sum(-x * smoothed_target, dim=1)
        return torch.mean(loss)


print("✓ Custom loss functions defined")

## 2. Utility Functions

In [ ]:
# Utility functions

class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def setup_seed(seed, cuda_deterministic=False):
    """Setup random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if cuda_deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


print("✓ Utility functions defined")

## 3. Configuration

In [ ]:
# Load config
from easydict import EasyDict as edict

config = edict()

# Model config
config.network = "swin_t"
config.img_size = 112
config.embedding_size = 512
config.fp16 = True

# Training config
config.optimizer = "adamw"
config.lr = 5e-4
config.weight_decay = 0.05
config.lr_name = 'cosine'
config.warmup_lr = 5e-7
config.min_lr = 5e-6
config.warmup_step = 2000
config.total_step = 60000
config.seed = 2048

# Data config
config.expression_train_dataset = "RAF-DB"
config.expression_val_dataset = "RAF-DB"
config.RAF_data = os.path.join(BASE_PATH, "dataset/RAF")
config.RAF_label = os.path.join(BASE_PATH, "dataset/list_patition_label.txt")
config.standard_train_sample_num = 1000000
config.batch_size = 32
config.train_num_workers = 2
config.train_pin_memory = True
config.val_batch_size = 64
config.val_num_workers = 0
config.val_pin_memory = True

# Augmentation config
config.INTERPOLATION = 'bicubic'
config.RAF_NUM_CLASSES = 7
config.RAF_LABEL_SMOOTHING = 0.1
config.AUG_COLOR_JITTER = 0.4
config.AUG_AUTO_AUGMENT = 'rand-m9-mstd0.5-inc1'
config.AUG_REPROB = 0.25
config.AUG_REMODE = 'pixel'
config.AUG_RECOUNT = 1
config.AUG_MIXUP = 0.0
config.AUG_CUTMIX = 0.0
config.AUG_CUTMIX_MINMAX = None
config.AUG_MIXUP_PROB = 1.0
config.AUG_MIXUP_SWITCH_PROB = 0.5
config.AUG_MIXUP_MODE = 'batch'
config.AUG_SCALE_SET = True
config.AUG_SCALE_SCALE = (1.0, 1.0)
config.AUG_SCALE_RATIO = (1.0, 1.0)

# Other config
config.output = OUTPUT_PATH
config.init = True
config.init_model = BASE_PATH
config.resume = False
config.resume_step = 0
config.save_all_states = True
config.verbose = 100
config.save_verbose = 500
config.frequent = 10
config.dali = False

# Setup seed
setup_seed(seed=config.seed, cuda_deterministic=False)

print("✓ Config loaded")
print(f"\nTraining Configuration:")
for key in ['network', 'batch_size', 'optimizer', 'lr', 'total_step', 'warmup_step']:
    print(f"  {key:20s}: {config[key]}")

## 4. Build Data Loaders

In [ ]:
# Build Data Loaders
from expression import get_analysis_train_dataloader, get_analysis_val_dataloader, ExpressionVerification
from backbones import get_model
from lr_scheduler import build_scheduler

print("\n" + "="*80)
print("BUILDING DATA LOADERS")
print("="*80)

# Create training dataloader
print("Creating training dataloader...")
expression_train_loader = get_analysis_train_dataloader("Expression", config)
print(f"✓ Training dataloader: {len(expression_train_loader)} batches")

# Create validation dataloader
print("Creating validation dataloader...")

if not distributed.is_initialized():
    from expression import RAFDBDataset
    from torch.utils.data import DataLoader
    from expression import SubsetRandomSampler

    dataset_val = RAFDBDataset(
        choose="test",
        data_path=config.RAF_data,
        label_path=config.RAF_label,
        img_size=config.img_size)
    indices = np.arange(len(dataset_val))
    sampler_val = SubsetRandomSampler(indices)
    expression_val_dataloader = DataLoader(
        dataset_val,
        sampler=sampler_val,
        batch_size=config.val_batch_size,
        shuffle=False,
        num_workers=config.val_num_workers,
        pin_memory=config.val_pin_memory,
        drop_last=False)
else:
    expression_val_dataloader = get_analysis_val_dataloader("Expression", config)

print(f"✓ Validation dataloader: {len(expression_val_dataloader)} batches")

# Check sample batch
for idx, batch in enumerate(expression_train_loader):
    img, label, img_aug = batch
    print(f"\nSample batch:")
    print(f"  Image shape: {img.shape}")
    print(f"  Label shape: {label.shape}")
    print(f"  Augmented image shape: {img_aug.shape}")
    print(f"  Batch size: {img.size(0)}")
    print(f"  Labels (first 5): {label[:5].tolist()}")
    break


## 5. Build Model

In [ ]:
# Build Model
print("\n" + "="*80)
print("BUILDING MODEL")
print("="*80)

# Get backbone
print("Loading backbone model...")
swin = get_model(config.network).cuda()
print(f"✓ Backbone loaded: {config.network}")

# Wrap with SwinTransFER
from expression import SwinTransFER
model = SwinTransFER(swin=swin, swin_num_features=768, num_classes=7, cam=True).cuda()
model.train()
print(f"✓ SwinTransFER model created and set to train mode")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

## 6. Build Optimizer & Scheduler

In [ ]:
# Build Optimizer and Scheduler
print("\n" + "="*80)
print("BUILDING OPTIMIZER & SCHEDULER")
print("="*80)

# Adjust LR based on batch size (for single GPU)
config.total_batch_size = config.batch_size
config.epoch_step = len(expression_train_loader)
config.num_epoch = math.ceil(config.total_step / config.epoch_step)

# Scale learning rate
config.lr = config.lr * config.total_batch_size / 512.0
config.warmup_lr = config.warmup_lr * config.total_batch_size / 512.0
config.min_lr = config.min_lr * config.total_batch_size / 512.0

print(f"Batch size scaling:")
print(f"  Total batch size: {config.total_batch_size}")
print(f"  Adjusted LR: {config.lr}")

# Create optimizer
if config.optimizer == "adamw":
    optimizer = torch.optim.AdamW(
        params=model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay)
    print(f"✓ Optimizer: AdamW (lr={config.lr:.6f}, weight_decay={config.weight_decay})")
elif config.optimizer == "sgd":
    optimizer = torch.optim.SGD(
        params=model.parameters(),
        lr=config.lr,
        momentum=0.9,
        weight_decay=config.weight_decay)
    print(f"✓ Optimizer: SGD")
else:
    raise ValueError(f"Unknown optimizer: {config.optimizer}")

# Create learning rate scheduler
lr_scheduler = build_scheduler(
    optimizer=optimizer,
    lr_name=config.lr_name,
    warmup_lr=config.warmup_lr,
    min_lr=config.min_lr,
    num_steps=config.total_step,
    warmup_steps=config.warmup_step)
print(f"✓ LR Scheduler: {config.lr_name} (warmup_steps={config.warmup_step})")

print(f"\nTraining plan:")
print(f"  Total steps: {config.total_step}")
print(f"  Epochs: {config.num_epoch}")
print(f"  Steps per epoch: {config.epoch_step}")
print(f"  Batch size: {config.batch_size}")
print(f"  Validation every {config.verbose} steps")

## 7. Initialize Training

In [ ]:
# Initialize training state
print("\n" + "="*80)
print("INITIALIZING TRAINING STATE")
print("="*80)

# Loss functions
expression_criteria = torch.nn.CrossEntropyLoss()
lsr2_loss = LSR2(0.3)
print(f"✓ Loss functions initialized")

# Gradient scaler for FP16
if config.fp16:
    amp_scaler = torch.cuda.amp.GradScaler(growth_interval=100)
    print(f"✓ Gradient scaler initialized (FP16 enabled)")
else:
    amp_scaler = None
    print(f"✓ FP16 disabled (full precision)")

# Training state
start_epoch = 0
global_step = 0
local_step = 0

# Load checkpoint if resuming
if config.init:
    init_path = os.path.join(config.init_model, "start_0.pt")
    if os.path.exists(init_path):
        print(f"Loading initial weights from {init_path}...")
        dict_checkpoint = torch.load(init_path, map_location='cuda')
        model.encoder.load_state_dict(dict_checkpoint["state_dict_backbone"], strict=False)
        print(f"✓ Initial weights loaded")
    else:
        print(f"⚠ Initial checkpoint not found at {init_path}")

# Logging
os.makedirs(config.output, exist_ok=True)
log_file = os.path.join(config.output, f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

# Metrics tracking
loss_meter = AverageMeter()

print(f"\n✓ Training state initialized")
print(f"  Start epoch: {start_epoch}")
print(f"  Global step: {global_step}")

## 8. Training Loop

In [ ]:
# Training Loop
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

train_history = {
    'global_step': [],
    'loss': [],
    'lr': [],
    'epoch': []
}

try:
    for epoch in range(start_epoch, config.num_epoch):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch + 1}/{config.num_epoch}")
        print(f"{'='*80}")
        
        loss_meter.reset()
        
        pbar = tqdm(enumerate(expression_train_loader), total=len(expression_train_loader))
        
        for idx, batch_data in pbar:
            # Skip if resuming
            if config.resume and idx < local_step:
                continue
            
            # Get batch data
            expression_img, expression_label, expression_img1 = batch_data
            expression_img = expression_img.cuda(non_blocking=True)
            expression_label = expression_label.cuda(non_blocking=True)
            expression_img1 = expression_img1.cuda(non_blocking=True)
            
            # Forward pass
            expression_output, hm = model(expression_img)
            expression_output1, hm1 = model(expression_img1)
            
            # Calculate loss with attention consistency
            grid_l = generate_flip_grid(7, 7).cuda(non_blocking=True)
            flip_loss = ACLoss(hm, hm1, grid_l, expression_output)
            
            flip_loss = flip_loss.mean(dim=-1).mean(dim=-1)  # N*7
            balance_weight = torch.tensor([0.95124031, 4.36690391, 1.71143654, 0.25714585,
                                          0.6191221, 1.74056738, 0.48617274]).cuda().view(7, 1)
            flip_loss = torch.mm(flip_loss, balance_weight).squeeze()
            
            # Total loss: LSR2 + weighted attention loss
            loss = lsr2_loss(expression_output, expression_label) + 0.1 * flip_loss.mean()
            
            # Backward pass
            if config.fp16:
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
                optimizer.step()
            
            optimizer.zero_grad()
            lr_scheduler.step_update(global_step)
            
            # Metrics
            loss_meter.update(loss.item(), expression_img.size(0))
            current_lr = optimizer.param_groups[0]["lr"]
            
            # Logging
            if (global_step + 1) % config.frequent == 0:
                train_history['global_step'].append(global_step)
                train_history['loss'].append(loss_meter.avg)
                train_history['lr'].append(current_lr)
                train_history['epoch'].append(epoch)
            
            # Progress bar
            pbar.set_description(
                f"Loss: {loss_meter.avg:.4f} | LR: {current_lr:.6f}"
            )
            
            # Validation and checkpoint
            if (global_step + 1) % config.verbose == 0:
                print(f"\nStep {global_step + 1}/{config.total_step} - Loss: {loss_meter.avg:.4f}")
            
            if config.save_all_states and (global_step + 1) % config.save_verbose == 0:
                checkpoint = {
                    "epoch": epoch,
                    "global_step": global_step,
                    "local_step": idx,
                    "state_dict_model": model.state_dict(),
                    "state_optimizer": optimizer.state_dict(),
                    "state_lr_scheduler": lr_scheduler.state_dict()
                }
                save_path = os.path.join(config.output, f"checkpoint_step_{global_step}.pt")
                torch.save(checkpoint, save_path)
                print(f"✓ Checkpoint saved: {save_path}")
            
            # Check if reached total steps
            if global_step >= config.total_step - 1:
                print(f"\n✓ Reached total steps ({config.total_step})")
                break
            else:
                global_step += 1
        
        if global_step >= config.total_step - 1:
            break
        
        print(f"Epoch {epoch + 1} completed - Average Loss: {loss_meter.avg:.4f}")

    print("\n" + "="*80)
    print("TRAINING COMPLETED")
    print("="*80)

except KeyboardInterrupt:
    print("\n⚠ Training interrupted by user")
except Exception as e:
    print(f"\n✗ Training error: {e}")
    import traceback
    traceback.print_exc()

## 9. Results & Visualization

In [ ]:
# Visualize training results
import matplotlib.pyplot as plt

print("\nGenerating training plots...")

if train_history['global_step']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Loss plot
    axes[0].plot(train_history['global_step'], train_history['loss'], 'b-', linewidth=2)
    axes[0].set_xlabel('Global Step', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Learning rate plot
    axes[1].plot(train_history['global_step'], train_history['lr'], 'r-', linewidth=2)
    axes[1].set_xlabel('Global Step', fontsize=12)
    axes[1].set_ylabel('Learning Rate', fontsize=12)
    axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(config.output, 'training_curves.png')
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    print(f"✓ Plot saved: {plot_path}")
    plt.show()

# Summary
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"Output directory: {config.output}")
print(f"Total steps: {global_step}")
print(f"Final loss: {loss_meter.avg:.4f}")
print(f"Files in output:")
for fname in sorted(os.listdir(config.output))[-5:]:
    fpath = os.path.join(config.output, fname)
    fsize = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {fname} ({fsize:.1f} MB)")